In [ ]:
# ============================================================
# MODEL REGISTRATION (PICKLE-FREE, SNOWFLAKE-SAFE)
# ============================================================

import json
import os
import tempfile
from datetime import datetime

import pandas as pd
from snowflake.snowpark import Session
from snowflake.snowpark.functions import col
from snowflake.ml.model import custom_model
from snowflake.ml.registry import Registry

# ------------------------------------------------------------
# Snowflake session
# ------------------------------------------------------------
session = get_active_session()

# ------------------------------------------------------------
# Custom ElasticNet Model (NO PICKLE)
# ------------------------------------------------------------
class ElasticNetInterceptModel(custom_model.CustomModel):

    def __init__(self, coefficients: dict, intercept: float):
        super().__init__(context=None)
        self.coefficients = coefficients
        self.intercept = intercept

    @custom_model.inference_api
    def predict(self, input: pd.DataFrame) -> pd.DataFrame:
        missing = set(self.coefficients) - set(input.columns)
        if missing:
            raise ValueError(f"Missing features: {missing}")

        prediction = sum(
            input[col] * coef for col, coef in self.coefficients.items()
        ) + self.intercept

        return pd.DataFrame({"prediction": prediction})


# ------------------------------------------------------------
# Initialize Registry
# ------------------------------------------------------------
reg = Registry(
    session=session,
    database_name="ORANGE_ZONE_SBX_TA",
    schema_name="REGISTRY"
)

# ------------------------------------------------------------
# Fetch intercept rows
# ------------------------------------------------------------
intercept_rows = (
    session.table("PUBLIC.PROD_FINAL_MODEL_LASSO_COEFFICIENTS")
    .filter(col("PARAMETERS") == "Intercept")
    .select(
        "REGIONNAME", "F_CODE", "COEFFICIENT_VALUE",
        "WMAPE", "MAPE", "RMSE", "BIAS", "TRACKING_SIGNAL",
        "ALPHA", "LAMBDA",
        "TRAIN_START_DATE", "TRAIN_END_DATE",
        "LOAD_TS"
    )
    .collect()
)

# ------------------------------------------------------------
# Process each model
# ------------------------------------------------------------
for row in intercept_rows:

    region = row["REGIONNAME"]
    f_code = row["F_CODE"]
    region_fcode = f"{region}_{f_code}"
    region_fcode_safe = region_fcode.replace("-", "_")

    print(f"Processing model: {region_fcode}")

    intercept = float(row["COEFFICIENT_VALUE"])

    # --------------------------------------------------------
    # Fetch coefficients
    # --------------------------------------------------------
    coef_rows = (
        session.table("PUBLIC.PROD_FINAL_MODEL_LASSO_COEFFICIENTS")
        .filter(
            (col("REGIONNAME") == region) &
            (col("F_CODE") == f_code) &
            (col("PARAMETERS") != "Intercept")
        )
        .select("PARAMETERS", "COEFFICIENT_VALUE")
        .collect()
    )

    coefficients = {
        r["PARAMETERS"]: float(r["COEFFICIENT_VALUE"])
        for r in coef_rows
    }

    features = sorted(coefficients.keys())

    # --------------------------------------------------------
    # Versioning logic (unchanged)
    # --------------------------------------------------------
    metadata_filename = f"model_metadata_{region_fcode_safe}.json"

    try:
        with open(metadata_filename, "r") as f:
            last_metadata = json.load(f)

        last_features = last_metadata["features"]
        last_period = last_metadata["train_period"]
        last_version = last_metadata.get("version", "v_0_0")

        major, minor = map(int, last_version.replace("v_", "").split("_"))

        if features != last_features:
            major += 1
            minor = 0
        elif (
            str(row["TRAIN_START_DATE"]) != last_period["start"]
            or str(row["TRAIN_END_DATE"]) != last_period["end"]
        ):
            minor += 1

        version = f"v_{major}_{minor}"

    except FileNotFoundError:
        version = "v_0_0"

    # --------------------------------------------------------
    # Metadata JSON
    # --------------------------------------------------------
    model_metadata = {
        "name": region_fcode,
        "model_type": "ElasticNet",
        "description": f"ElasticNet demand model for {region_fcode}",
        "version": version,
        "metrics": {
            "wmape": float(row["WMAPE"]),
            "mape": float(row["MAPE"]),
            "rmse": float(row["RMSE"]),
            "bias": float(row["BIAS"]),
            "tracking_signal": float(row["TRACKING_SIGNAL"]),
        },
        "hyperparameters": {
            "alpha": float(row["ALPHA"]),
            "lambda": float(row["LAMBDA"]),
        },
        "train_period": {
            "start": str(row["TRAIN_START_DATE"]),
            "end": str(row["TRAIN_END_DATE"]),
        },
        "features": features,
        "training_date": str(row["LOAD_TS"]),
        "tags": {
            "region": region,
            "f_code": f_code,
            "algorithm": "ElasticNet",
        },
        "owner": "tiger_analytics_team",
        "training_wh": "ML_TRAIN_WH",
        "artifact_stage": "@ML_PROD.CODE/retention_v1.2.0",
    }

    tmp_dir = tempfile.gettempdir()
    artifact_path = os.path.join(tmp_dir, metadata_filename)

    with open(artifact_path, "w") as f:
        json.dump(model_metadata, f, indent=2)

    # --------------------------------------------------------
    # Create Custom Model
    # --------------------------------------------------------
    model_instance = ElasticNetInterceptModel(
        coefficients=coefficients,
        intercept=intercept
    )

    model_name = f"intercept_model_{region_fcode_safe}"

    # --------------------------------------------------------
    # Log Model
    # --------------------------------------------------------
    mv = reg.log_model(
        model=model_instance,
        model_name=model_name,
        conda_dependencies=["pandas"],
        options={"relax_version": False},
        user_files={"metadata": [artifact_path]},
        comment=f"ElasticNet model for {region_fcode}",
        sample_input_data=pd.DataFrame(
            {k: [0.0] for k in features}
        ),
    )

    # --------------------------------------------------------
    # Alias Management
    # --------------------------------------------------------
    model_ref = reg.get_model(model_name)

    for v in model_ref.versions():
        if v.version_name != mv.version_name:
            try:
                v.unset_alias("PROD")
                v.unset_alias("ARCHIVED")
            except Exception:
                pass
            v.set_alias("ARCHIVED")

    mv.set_alias("PROD")

    print(
        f"Model {model_name} | Registry {mv.version_name} | "
        f"Semantic {version} → PROD"
    )
